# Guard generated answers with retrieved evidence

`GroundedAnswerPolicy` applies deterministic checks after retrieval and generation. The examples below show each production decision without requiring a model download.

In [1]:
import com.integrallis.models.rag.GroundedAnswer;
import com.integrallis.models.rag.GroundedAnswerPolicy;
import com.integrallis.models.rag.GroundingDocument;
import java.util.List;

In [2]:
var settlement = new GroundingDocument(
    "payments-settlement",
    "Payment settlement",
    "Approved domestic ACH claims settle within 2 business days. "
        + "International claim wires settle within 5 business days.",
    8.4f,
    1);
var weak = new GroundingDocument(
    "claims-auto-glass",
    "Auto glass claims",
    "Windshield repair has a 75 dollar deductible.",
    1.2f,
    1);
var policy = new GroundedAnswerPolicy(2.0f);

## Compare grounding decisions

Trusted citations are retained, supported text can receive a derived citation, weak retrieval abstains, and unsupported text falls back to the retrieved source.

In [3]:
GroundedAnswer cited = policy.apply(
    "How long do both payment types take?",
    List.of(settlement),
    "Domestic claims take 2 days and international wires take 5 days. "
        + "[payments-settlement]");
GroundedAnswer derived = policy.apply(
    "How long do both payment types take?",
    List.of(settlement),
    "Domestic claims settle within 2 business days.");
GroundedAnswer weakRetrieval = policy.apply(
    "What deductible applies to a lunar rover?",
    List.of(weak),
    "The deductible is 75 dollars. [claims-auto-glass]");
GroundedAnswer unsupported = policy.apply(
    "How long do both payment types take?",
    List.of(settlement),
    "Both payment types settle instantly.");

System.out.println("cited: " + cited.decision());
System.out.println("derived: " + derived.decision());
System.out.println("weak retrieval: " + weakRetrieval.decision());
System.out.println("unsupported: " + unsupported.decision());
System.out.println("fallback text: " + unsupported.text());

cited: MODEL_ANSWER


derived: MODEL_ANSWER_WITH_DERIVED_CITATIONS


weak retrieval: RETRIEVAL_ABSTENTION


unsupported: EXTRACTIVE_FALLBACK


fallback text: Approved domestic ACH claims settle within 2 business days. International claim wires settle within 5 business days. [payments-settlement]
